<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [11]:
# TODO: implemente load_data
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def load_data(seed=42):

    fashion_mnist = fetch_openml('mnist_784', version=1, as_frame=False)
    X, y = fashion_mnist.data, fashion_mnist.target
    
    y = y.astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    return X_train, X_test, y_train, y_test

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
# TODO: implemente train_random_forest e train_adaboost

def train_random_forest(X_train, y_train, seed=42):
    """
    Treina Random Forest com hiperparâmetros otimizados para Fashion MNIST
    """
    rf = RandomForestClassifier(
        n_estimators=200,  
        max_depth=20,      
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=seed,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    return rf

def train_adaboost(X_train, y_train, seed=42):
    """
    Treina AdaBoost com hiperparâmetros otimizados para Fashion MNIST
    """
    from sklearn.tree import DecisionTreeClassifier
    
    weak_learner = DecisionTreeClassifier(
        max_depth=4,         
        min_samples_split=10,
        random_state=seed
    )
    
    ab = AdaBoostClassifier(
        estimator=weak_learner,
        n_estimators=200,
        learning_rate=0.5,
        random_state=seed
    )
    ab.fit(X_train, y_train)
    return ab

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [13]:
# TODO: implemente evaluate

def evaluate(model, X_test, y_test):
    """
    Avalia o modelo e retorna um dicionário com todas as métricas
    """
    y_pred = model.predict(X_test)
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
        'f1': f1_score(y_test, y_pred, average='weighted')
    }
    
    return metrics

def evaluate_complete(model, X_test, y_test):
    """
    Função auxiliar para avaliação completa (mantida para análises detalhadas)
    """
    return evaluate(model, X_test, y_test)

**Adicione seu texto de solução aqui.**

Para os modelos de ensemble (Random Forest e AdaBoost) utilizados nesta atividade, **a normalização não é estritamente necessária**. Justificativa:

1. **Random Forest**: Baseado em árvores de decisão, que fazem divisões baseadas em limiares dos valores das features. A escala das features não afeta o desempenho das árvores, pois elas operam com comparações de "maior/menor que" em vez de distâncias euclidianas.

2. **AdaBoost**: Também utiliza árvores de decisão como weak learners, portanto tem a mesma característica do Random Forest em relação à escala das features.

No entanto, é importante considerar que:

- **Consistência**: Para comparações com outros algoritmos (como SVM, redes neurais), a normalização seria essencial
- **Boa prática**: Em pipelines de machine learning production, a normalização é geralmente incluída para manter consistência
- **Performance computacional**: Features com escalas muito diferentes podem afetar a eficiência computacional

Para este experimento específico com os modelos solicitados, os resultados devem ser muito similares com ou sem normalização, mas a inclusão da normalização tornaria o pipeline mais robusto e reutilizável.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [14]:
# TODO: implemente run_pipeline

def run_pipeline(model_type="rf", seed=42):
    """
    Executa o pipeline completo de treinamento e avaliação
    Retorna a acurácia para compatibilidade com testes
    """
    X_train, X_test, y_train, y_test = load_data(seed)
    
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")
    
    metrics = evaluate(model, X_test, y_test)
    
    return metrics['accuracy']

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

**Sobre o overfitting:**
O overfitting em Random Forest geralmente começa a ocorrer quando as árvores individuais se tornam muito profundas e complexas. Em Random Forest, o efeito é atenuado pelo ensemble, mas ainda pode ocorrer quando:
- max_depth é muito alto (acima de 20-30 para datasets como Fashion MNIST)
- min_samples_split é muito baixo (abaixo de 5)
- min_samples_leaf é muito baixo (abaixo de 2)

O ponto exato onde o overfitting começa depende da complexidade do dataset e da quantidade de dados disponíveis. Para Fashion MNIST, geralmente começamos a ver sinais de overfitting com max_depth > 25.

**Por que 100% no treino com max_depth=None:**
Quando max_depth=None, as árvores de decisão crescem até que cada folha seja "pura" (contenha amostras de uma única classe) ou até que min_samples_split seja atingido. Isso resulta em:

1. **Memorização completa**: A árvore aprende perfeitamente os dados de treinamento, criando regras específicas para cada amostra individual.

2. **Capacidade de separação perfeita**: Como o Fashion MNIST é um dataset com 784 features (pixels), uma árvore suficientemente profunda pode criar caminhos que separam perfeitamente cada classe nos dados de treino, mesmo que essas separações não generalizem.

3. **Overfitting garantido**: Essa capacidade perfeita nos dados de treino geralmente não generaliza para dados novos, resultando em desempenho significativamente inferior no conjunto de teste.

O Random Forest mitiga esse problema através do bootstrap (amostragem com reposição) e seleção aleatória de features, mas o overfitting ainda pode ocorrer se as árvores individuais forem muito complexas.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [15]:
# TODO: implementar comparação completa dos modelos

X_train, X_test, y_train, y_test = load_data(42)

rf_model = train_random_forest(X_train, y_train, 42)
ab_model = train_adaboost(X_train, y_train, 42)

# Avaliação completa dos modelos
rf_metrics = evaluate_complete(rf_model, X_test, y_test)
ab_metrics = evaluate_complete(ab_model, X_test, y_test)

print("=== Random Forest ===")
print(f"Acurácia: {rf_metrics['accuracy']:.4f}")
print(f"Precisão: {rf_metrics['precision']:.4f}")
print(f"Recall: {rf_metrics['recall']:.4f}")
print(f"F1-Score: {rf_metrics['f1']:.4f}")

print("\n=== AdaBoost ===")
print(f"Acurácia: {ab_metrics['accuracy']:.4f}")
print(f"Precisão: {ab_metrics['precision']:.4f}")
print(f"Recall: {ab_metrics['recall']:.4f}")
print(f"F1-Score: {ab_metrics['f1']:.4f}")

print("\n=== Comparação ===")
print(f"Diferença na Acurácia: {abs(rf_metrics['accuracy'] - ab_metrics['accuracy']):.4f}")
print(f"Melhor modelo (acurácia): {'Random Forest' if rf_metrics['accuracy'] > ab_metrics['accuracy'] else 'AdaBoost'}")
print(f"Melhor modelo (F1-Score): {'Random Forest' if rf_metrics['f1'] > ab_metrics['f1'] else 'AdaBoost'}")

=== Random Forest ===
Acurácia: 0.9646
Precisão: 0.9647
Recall: 0.9646
F1-Score: 0.9646

=== AdaBoost ===
Acurácia: 0.9413
Precisão: 0.9415
Recall: 0.9413
F1-Score: 0.9413

=== Comparação ===
Diferença na Acurácia: 0.0234
Melhor modelo (acurácia): Random Forest
Melhor modelo (F1-Score): Random Forest


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
# TODO: implementar teste de reprodutibilidade

print("=== Teste de Reprodutibilidade ===")

rf_acc_42_v1 = run_pipeline("rf", seed=42)
rf_acc_42_v2 = run_pipeline("rf", seed=42)
rf_acc_7 = run_pipeline("rf", seed=7)

ab_acc_42_v1 = run_pipeline("ab", seed=42)
ab_acc_42_v2 = run_pipeline("ab", seed=42)
ab_acc_7 = run_pipeline("ab", seed=7)

print("Random Forest (seed=42):")
print(f"  Execução 1 - Acurácia: {rf_acc_42_v1:.6f}")
print(f"  Execução 2 - Acurácia: {rf_acc_42_v2:.6f}")
print(f"  Diferença: {abs(rf_acc_42_v1 - rf_acc_42_v2):.10f}")

print("\nAdaBoost (seed=42):")
print(f"  Execução 1 - Acurácia: {ab_acc_42_v1:.6f}")
print(f"  Execução 2 - Acurácia: {ab_acc_42_v2:.6f}")
print(f"  Diferença: {abs(ab_acc_42_v1 - ab_acc_42_v2):.10f}")

print("\nComparação entre seeds diferentes:")
print(f"Random Forest (seed=42 vs seed=7): {abs(rf_acc_42_v1 - rf_acc_7):.6f}")
print(f"AdaBoost (seed=42 vs seed=7): {abs(ab_acc_42_v1 - ab_acc_7):.6f}")

rf_reprodutivel = abs(rf_acc_42_v1 - rf_acc_42_v2) < 1e-6
ab_reprodutivel = abs(ab_acc_42_v1 - ab_acc_42_v2) < 1e-6

print(f"\nO experimento é reprodutível? {'SIM' if rf_reprodutivel and ab_reprodutivel else 'NÃO'}")
print("Justificativa: Mesma seed produz exatamente o mesmo resultado, garantindo reprodutibilidade.")

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
# TODO: implementar análise de overfitting

print("=== Análise de Overfitting ===")

X_train, X_test, y_train, y_test = load_data(42)

rf_model = train_random_forest(X_train, y_train, 42)
ab_model = train_adaboost(X_train, y_train, 42)

rf_train_metrics = evaluate_complete(rf_model, X_train, y_train)
ab_train_metrics = evaluate_complete(ab_model, X_train, y_train)

rf_test_metrics = evaluate_complete(rf_model, X_test, y_test)
ab_test_metrics = evaluate_complete(ab_model, X_test, y_test)

print("Random Forest:")
print(f"  Acurácia Treino: {rf_train_metrics['accuracy']:.4f}")
print(f"  Acurácia Teste: {rf_test_metrics['accuracy']:.4f}")
print(f"  Diferença (Overfitting): {rf_train_metrics['accuracy'] - rf_test_metrics['accuracy']:.4f}")

print("\nAdaBoost:")
print(f"  Acurácia Treino: {ab_train_metrics['accuracy']:.4f}")
print(f"  Acurácia Teste: {ab_test_metrics['accuracy']:.4f}")
print(f"  Diferença (Overfitting): {ab_train_metrics['accuracy'] - ab_test_metrics['accuracy']:.4f}")

rf_overfit = rf_train_metrics['accuracy'] - rf_test_metrics['accuracy']
ab_overfit = ab_train_metrics['accuracy'] - ab_test_metrics['accuracy']

print(f"\nExiste overfitting? {'SIM' if (rf_overfit > 0.05) or (ab_overfit > 0.05) else 'NÃO'}")
print(f"Modelo com mais overfitting: {'Random Forest' if rf_overfit > ab_overfit else 'AdaBoost'}")

if rf_overfit > ab_overfit:
    print("\nRandom Forest tende a sofrer mais com overfitting, pois as árvores individuais podem se tornar muito complexas.")
else:
    print("\nAdaBoost tende a sofrer mais com overfitting, pois foca em amostras difíceis e pode memorizar ruído.")

=== Análise de Overfitting ===
Random Forest:
  Acurácia Treino: 1.0000
  Acurácia Teste: 0.8819
  Diferença (Overfitting): 0.1181

AdaBoost:
  Acurácia Treino: 0.5757
  Acurácia Teste: 0.5729
  Diferença (Overfitting): 0.0028

Existe overfitting? SIM
Modelo com mais overfitting: Random Forest

Random Forest tende a sofrer mais com overfitting, pois as árvores individuais podem se tornar muito complexas.


# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
# TODO: implementar variação de hiperparâmetros

print("=== Variação de Hiperparâmetros ===")

n_estimators_list = [50, 100, 200, 300]

print("Random Forest - Variação de n_estimators:")
rf_results = []
for n_est in n_estimators_list:
    rf = RandomForestClassifier(n_estimators=n_est, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    acc = rf.score(X_test, y_test)
    rf_results.append(acc)
    print(f"  n_estimators={n_est}: Acurácia = {acc:.4f}")

print("\nAdaBoost - Variação de n_estimators:")
ab_results = []
for n_est in n_estimators_list:
    from sklearn.tree import DecisionTreeClassifier
    weak_learner = DecisionTreeClassifier(max_depth=3, random_state=42)
    ab = AdaBoostClassifier(estimator=weak_learner, n_estimators=n_est, random_state=42)
    ab.fit(X_train, y_train)
    acc = ab.score(X_test, y_test)
    ab_results.append(acc)
    print(f"  n_estimators={n_est}: Acurácia = {acc:.4f}")

rf_std = np.std(rf_results)
ab_std = np.std(ab_results)

print(f"\nAnálise da Sensibilidade:")
print(f"Random Forest - Desvio Padrão: {rf_std:.4f}")
print(f"AdaBoost - Desvio Padrão: {ab_std:.4f}")

print(f"\nModelo mais sensível: {'Random Forest' if rf_std > ab_std else 'AdaBoost'}")

if rf_std > ab_std:
    print("Random Forest é mais sensível a mudanças em n_estimators.")
else:
    print("AdaBoost é mais sensível a mudanças em n_estimators.")

best_rf_idx = np.argmax(rf_results)
best_ab_idx = np.argmax(ab_results)

print(f"\nMelhor configuração:")
print(f"Random Forest: n_estimators={n_estimators_list[best_rf_idx]} (acurácia: {rf_results[best_rf_idx]:.4f})")
print(f"AdaBoost: n_estimators={n_estimators_list[best_ab_idx]} (acurácia: {ab_results[best_ab_idx]:.4f})")

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

**Qual modelo é mais sensível a mudanças?**

Baseado na análise de variação de hiperparâmetros:

**AdaBoost é mais sensível a mudanças em n_estimators** do que Random Forest. Isso ocorre porque:

1. **AdaBoost**: Pequenas mudanças no número de estimadores podem afetar significativamente o desempenho, pois cada estimador subsequente depende dos erros dos anteriores. Mais estimadores podem levar a overfitting focado em amostras difíceis.

2. **Random Forest**: É mais estável porque cada árvore é independente e treinada em bootstrap samples. Adicionar mais árvores geralmente melhora ou mantém o desempenho, com menor risco de degradação.

**Desempenho muda significativamente?**
- **Random Forest**: Mudanças mais graduais e previsíveis
- **AdaBoost**: Mudanças mais abruptas e menos previsíveis

Portanto, AdaBoost requer mais cuidado na seleção de hiperparâmetros, enquanto Random Forest é mais robusto a variações.


**1. A acurácia é suficiente para avaliar os modelos?**

Não, a acurácia não é suficiente para avaliar completamente os modelos, especialmente em datasets multiclasse como o Fashion MNIST. Embora seja uma métrica importante, ela não captura aspectos cruciais como o desempenho por classe, a taxa de falsos positivos/negativos, e o balanceamento entre precisão e recall. Em cenários do mundo real, diferentes erros de classificação podem ter custos diferentes, e métricas como F1-score, matriz de confusão e curvas ROC fornecem uma visão mais completa do comportamento do modelo.

**2. Como você garante que o resultado não ocorreu por acaso?**

A garantia contra resultados aleatórios é obtida através da reprodutibilidade controlada e validação estatística. Primeiro, fixamos todas as sementes aleatórias (random_state) para garantir que os resultados sejam consistentes entre execuções. Segundo, utilizamos separação estratificada dos dados para manter a distribuição das classes entre treino e teste. Terceiro, poderíamos implementar validação cruzada para reduzir a variabilidade devido à particionamento específico dos dados. Finalmente, a comparação entre múltiplos modelos e diferentes configurações de hiperparâmetros ajuda a confirmar que as diferenças observadas são consistentes e não flutuações aleatórias.

**3. Cite dois possíveis problemas metodológicos neste experimento.**

Primeiro, o uso de apenas uma partição treino/teste (80/20) pode não ser suficiente para avaliar robustamente o desempenho dos modelos, pois os resultados podem depender especificamente de como os dados foram divididos. Segundo, a otimização de hiperparâmetros foi feita de forma limitada (apenas n_estimators), sem uma busca sistemática ou validação cruzada, o que pode levar a conclusões incompletas sobre o potencial real dos modelos.

**4. O pipeline implementado é confiável? Justifique.**

O pipeline implementado é parcialmente confiável, mas tem limitações. É confiável em termos de reprodutibilidade e estrutura básica, com todas as funções implementadas corretamente e resultados consistentes para mesmas sementes. No entanto, não é totalmente confiável para produção ou conclusões definitivas porque: (1) não inclui validação cruzada para avaliação mais robusta, (2) a otimização de hiperparâmetros é superficial, (3) não avalia a estabilidade dos modelos frente a diferentes partições dos dados, e (4) não considera outras métricas importantes além da acurácia. Para maior confiabilidade, seriam necessários testes mais extensivos e uma metodologia de validação mais rigorosa.